# 04 模型模块 (core.models)

覆盖 LogisticRegression、ScoreCard、自定义损失函数、评分漂移检测。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import hscredit as hsc
from hscredit import (germancredit, init_setting, OptimalBinning,
    WOEEncoder, LogisticRegression, ScoreCard,
    FocalLoss, WeightedBCELoss, KSMetric, GiniMetric,
    XGBoostLossAdapter, LightGBMLossAdapter,
)
from sklearn.model_selection import train_test_split
from hscredit.core.metrics import ks, auc
init_setting()
df = germancredit()
target = 'creditability'
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
X = df[num_feats]; y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

## 1. LogisticRegression（hscredit版，含统计量）

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=1000, calculate_stats=True)
lr.fit(X_tr, y_tr)
proba = lr.predict_proba(X_te)[:, 1]
print(f'KS: {ks(y_te, proba):.4f}, AUC: {auc(y_te, proba):.4f}')
print('\n系数摘要:')
print(lr.summary().head(5))

## 2. ScoreCard（评分卡）

In [ ]:
# 先做WOE编码
woe = WOEEncoder(max_n_bins=5)
X_woe_tr = woe.fit_transform(X_tr, y_tr)
X_woe_te = woe.transform(X_te)

sc = ScoreCard(
    pdo=20, score=600, odds=1/19,
    lr_kwargs={'C': 1.0, 'max_iter': 1000, 'calculate_stats': True}
)
sc.fit(X_woe_tr, y_tr, woe_encoder=woe)
scores = sc.predict(X_woe_te)
print('评分样例:', scores[:10].tolist())
print('评分分布:', pd.Series(scores).describe().round(1))

In [ ]:
# 查看评分卡表

In [ ]:
sc_table = sc.scorecard_table_
print(sc_table.columns.tolist())
sc_table.head(10)

## 3. 自定义损失函数

In [ ]:
# Focal Loss（处理样本不平衡）
fl = FocalLoss(gamma=2.0, alpha=0.25)
print('FocalLoss:', fl)

# WeightedBCE
wbce = WeightedBCELoss(pos_weight=3.0)
print('WeightedBCE:', wbce)

# 评估指标
ks_m = KSMetric()
gini_m = GiniMetric()
print('KSMetric:', ks_m)
print('GiniMetric:', gini_m)

## 4. XGBoost/LightGBM 损失适配器

In [ ]:
try:
    import xgboost as xgb
    adapter = XGBoostLossAdapter(FocalLoss(gamma=2.0))
    print('XGBoost FocalLoss adapter:', adapter)
except ImportError:
    print('xgboost 未安装')
try:
    import lightgbm as lgb
    lgb_adapter = LightGBMLossAdapter(WeightedBCELoss(pos_weight=3.0))
    print('LightGBM WeightedBCE adapter:', lgb_adapter)
except ImportError:
    print('lightgbm 未安装')

## 5. 概率转评分 (probability_to_score)

In [ ]:
from hscredit.core.models.probability_to_score import probability_to_score
proba_sample = lr.predict_proba(X_te)[:, 1]
scores_direct = probability_to_score(proba_sample, pdo=20, score=600, odds=1/19)
print('概率转评分:', scores_direct[:5])